# Imports

In [16]:
import os
import time
import requests
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sklearn.decomposition import PCA
from sklearn.random_projection import GaussianRandomProjection

# Configuration

In [17]:

NTFY_TOPIC = "aysel_tfe_server_9988" 
# "Flickr8k"  or "Flickr30k" or "ConceptualCaptions"
CURRENT_DATASET = "ConceptualCaptions"  

def send_ntfy(message, title="TFE Production"):
    try:
        requests.post(f"https://ntfy.sh/{NTFY_TOPIC}", 
                      data=message.encode('utf-8'),
                      headers={"Title": title, "Priority": "3"})
    except: pass

In [18]:
DATA_DIR = os.path.join(os.getcwd(), 'TFE_Data')
RAW_DIR = os.path.join(DATA_DIR, 'Features_RAW')
REDUCED_DIR = os.path.join(DATA_DIR, 'Features_Reduced')

os.makedirs(REDUCED_DIR, exist_ok=True)

# Data Loading

In [19]:
def load_raw_matrix(modality, model_name):
    """Loads a matrix from the Features_RAW subfolder."""
    file_path = os.path.join(RAW_DIR, f"X_{modality}_{model_name}_raw_{CURRENT_DATASET}.npy")
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"Missing RAW matrix: {file_path}")
    return np.load(file_path)

# Dimension Reduction

In [20]:
def run_production_batch(modality, model_name):
    print(f"\nProcessing: {model_name.upper()} ({modality})")
    
    try:
        X_raw = load_raw_matrix(modality, model_name)
    except Exception as e:
        print(f"Skipping {model_name}: {e}")
        return

    input_dim = X_raw.shape[1]
    
    for method_name, reduction_func in REDUCTION_PIPELINE.items():
        for dim in DIMENSIONS_TO_TEST:
            # Skip if target dimension is larger than original
            if dim >= input_dim: continue
            
            # 1. Apply Reduction
            X_reduced = reduction_func(X_raw, dim)
            
            # 2. Save with Standardized Naming Convention
            save_filename = f"X_{modality}_{model_name}_{method_name}_{dim}_{CURRENT_DATASET}.npy"
            save_path = os.path.join(REDUCED_DIR, save_filename)
            np.save(save_path, X_reduced)
            
    print(f"Finished all reductions for {model_name}")
    send_ntfy(f"Reductions for {model_name} ({CURRENT_DATASET}) are saved to disk.")

# Execution

## Parameters

In [21]:
DIMENSIONS_TO_TEST = [512, 256, 128, 64, 32, 16]

REDUCTION_PIPELINE = {
    "pca": lambda mat, dim: PCA(n_components=dim).fit_transform(mat),
    "grp": lambda mat, dim: GaussianRandomProjection(n_components=dim).fit_transform(mat)
}

VISION_MODELS = ["resnet50", "mobilenet_v3", "vit", "deit", "clip_vision"]
TEXT_MODELS = ["rnn", "bert", "roberta", "gpt2", "clip_text"]

## Run Experiment

In [22]:
print(f"Starting Dimensionality Reduction for {CURRENT_DATASET}...")

# Run Vision
for v_model in VISION_MODELS:
    run_production_batch("vision", v_model)

# Run Text
for t_model in TEXT_MODELS:
    run_production_batch("text", t_model)

print("\n ALL REDUCTIONS COMPLETE.")

Starting Dimensionality Reduction for ConceptualCaptions...

Processing: RESNET50 (vision)
Finished all reductions for resnet50

Processing: MOBILENET_V3 (vision)
Finished all reductions for mobilenet_v3

Processing: VIT (vision)
Finished all reductions for vit

Processing: DEIT (vision)
Finished all reductions for deit

Processing: CLIP_VISION (vision)
Finished all reductions for clip_vision

Processing: RNN (text)
Finished all reductions for rnn

Processing: BERT (text)
Finished all reductions for bert

Processing: ROBERTA (text)
Finished all reductions for roberta

Processing: GPT2 (text)
Finished all reductions for gpt2

Processing: CLIP_TEXT (text)
Finished all reductions for clip_text

 ALL REDUCTIONS COMPLETE.
